In [2]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("03b_gold_preview").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")

BUCKET = "taxis-chicago-grandes-datos-506301"
GOLD = f"gs://{BUCKET}/gold"
EXPORT = f"{GOLD}/app_export"

In [3]:
def preview(title, df, sample=None, n=5):
    """Imprime tamaño y columnas de la tabla, y muestra unas filas de ejemplo."""
    print(f"\n=== {title} ===")
    print(f"Filas: {df.count():,} | Columnas: {len(df.columns)}")
    print(", ".join(f"{c} ({t})" for c, t in df.dtypes))
    display((df if sample is None else sample).limit(n).toPandas())


def read_csv(name):
    return spark.read.option("header", True).option("inferSchema", True).csv(f"{EXPORT}/{name}")

In [4]:
# --- Tablas de gold (Parquet) ---
demand = spark.read.parquet(f"{GOLD}/zone_hour_demand")
preview("zone_hour_demand: el Loop, viernes 6 de octubre de 2023, de 16 a 20 h", demand,
        demand.filter((F.col("zone") == 32) & (F.col("date") == "2023-10-06") & (F.col("hour") >= 16))
              .orderBy("hour"))

kpis = spark.read.parquet(f"{GOLD}/daily_kpis")
preview("daily_kpis: 6 de octubre de 2023, combinaciones con más viajes", kpis,
        kpis.filter(F.col("date") == "2023-10-06").orderBy(F.desc("trips")))

zones = spark.read.parquet(f"{GOLD}/dim_zones")
preview("dim_zones: primeras zonas", zones, zones.orderBy("zone"))


=== zone_hour_demand: el Loop, viernes 6 de octubre de 2023, de 16 a 20 h ===


Filas: 7,371,672 | Columnas: 17
date (date), zone (int), hour (int), zone_name (string), trips (bigint), revenue (double), active_taxis (bigint), avg_minutes (double), month (int), day_of_week (int), is_weekend (int), is_holiday (int), temp_c (double), prcp_mm (double), snow (int), profile_4w (double), year (int)


,date,zone,hour,zone_name,trips,revenue,active_taxis,avg_minutes,month,day_of_week,is_weekend,is_holiday,temp_c,prcp_mm,snow,profile_4w,year
0,2023-10-06,32,16,Loop,344,6334.97,286,22.5,10,6,0,0,14.3,5.3,0,280.25,2023
1,2023-10-06,32,17,Loop,372,6450.31,311,21.1,10,6,0,0,14.3,5.3,0,283.75,2023
2,2023-10-06,32,18,Loop,324,5253.14,267,18.0,10,6,0,0,14.3,5.3,0,257.25,2023
3,2023-10-06,32,19,Loop,240,3661.54,200,15.4,10,6,0,0,14.3,5.3,0,179.50,2023
4,2023-10-06,32,20,Loop,127,1878.94,111,12.6,10,6,0,0,14.3,5.3,0,123.50,2023



=== daily_kpis: 6 de octubre de 2023, combinaciones con más viajes ===


Filas: 379,022 | Columnas: 7
date (date), company (string), payment_type (string), trips (bigint), revenue (double), avg_minutes (double), active_taxis (bigint)


,date,company,payment_type,trips,revenue,avg_minutes,active_taxis
0,2023-10-06,Taxi Affiliation Services,Credit Card,1825,66668.61,27.4,400
1,2023-10-06,Taxi Affiliation Services,Cash,1286,26224.95,18.8,371
2,2023-10-06,Taxicab Insurance Agency Llc,Credit Card,1204,47619.46,29.9,262
3,2023-10-06,Sun Taxi,Credit Card,1196,47215.44,29.8,268
4,2023-10-06,Flash Cab,Cash,1143,28093.60,21.7,314



=== dim_zones: primeras zonas ===


Filas: 77 | Columnas: 4
zone (int), zone_name (string), lat (double), lon (double)


,zone,zone_name,lat,lon
0,1,Rogers Park,42.009380,-87.670183
1,2,West Ridge,42.001501,-87.694939
2,3,Uptown,41.965221,-87.655840
3,4,Lincoln Square,41.974616,-87.687338
4,5,North Center,41.948177,-87.683344


In [5]:
# --- Exportaciones para la app (CSV) ---
daily = read_csv("dashboard_daily")
preview("dashboard_daily: últimos días", daily, daily.orderBy(F.desc("date")))

zone_hour = read_csv("dashboard_zone_hour")
preview("dashboard_zone_hour: el Loop, viernes de 2023, de 16 a 20 h", zone_hour,
        zone_hour.filter((F.col("year") == 2023) & (F.col("zone") == 32)
                         & (F.col("day_of_week") == 6) & (F.col("hour") >= 16))
                 .orderBy("hour"))

companies = read_csv("dashboard_companies")
preview("dashboard_companies: empresas con más viajes en 2023", companies,
        companies.filter(F.col("year") == 2023).orderBy(F.desc("trips")))


=== dashboard_daily: últimos días ===
Filas: 3,989 | Columnas: 6
date (timestamp), trips (int), revenue (double), temp_c (double), prcp_mm (double), snow (int)


,date,trips,revenue,temp_c,prcp_mm,snow
0,2023-12-31,10063,268723.86,1.1,0.0,1
1,2023-12-30,10490,288875.73,-0.3,3.6,0
2,2023-12-29,12609,321869.74,3.9,6.9,0
3,2023-12-28,13008,327487.30,3.1,0.0,0
4,2023-12-27,12346,330727.05,2.4,0.3,0



=== dashboard_zone_hour: el Loop, viernes de 2023, de 16 a 20 h ===


Filas: 142,296 | Columnas: 7
year (int), zone (int), zone_name (string), day_of_week (int), hour (int), trips (int), revenue (double)


,year,zone,zone_name,day_of_week,hour,trips,revenue
0,2023,32,Loop,6,16,13904,223824.67
1,2023,32,Loop,6,17,14099,212256.44
2,2023,32,Loop,6,18,12702,179462.31
3,2023,32,Loop,6,19,9788,134337.99
4,2023,32,Loop,6,20,6318,86858.40



=== dashboard_companies: empresas con más viajes en 2023 ===
Filas: 638 | Columnas: 4
year (int), company (string), trips (int), revenue (double)


,year,company,trips,revenue
0,2023,Flash Cab,1135360,28354391.68
1,2023,Taxi Affiliation Services,1119057,31471124.12
2,2023,Sun Taxi,680355,19581905.13
3,2023,City Service,609915,16933836.92
4,2023,Taxicab Insurance Agency Llc,571063,16584699.15
